In [ ]:
import sys
import pandas as pd
from pathlib import Path

from PySide6.QtWidgets import (
    QApplication, QDialog, QVBoxLayout, QLabel, QTableWidget, QTableWidgetItem,
    QPushButton, QMessageBox
)
from PySide6.QtCore import Qt


# =========================
# 엑셀 열 매핑(헤더 없음)
# =========================
# 0-based index
COL_A_NAME = 0   # A열: 이름
COL_C_PLATFORM = 2  # C열: 플랫폼명
COL_D_PRODUCT = 3   # D열: 제품코드(너가 말한 "A와 D가 같은데"의 D)
COL_K_ORDERNO = 10  # K열: 주문코드


def _clean_series(s: pd.Series) -> pd.Series:
    return s.astype(str).replace("nan", "").str.strip()


def check_merge_forbidden(excel_path: str) -> pd.DataFrame:
    """
    합포장 금지 검수:
    - A(이름) + D(제품코드)가 같은 그룹에서
    - K(주문코드)가 서로 다른 값이 2개 이상이면 문제로 판단

    반환: 문제 행들만 모은 DataFrame (표시용: A, C, K)
    """
    df = pd.read_excel(excel_path, header=None, dtype=str)

    # 파일에 열이 충분한지 체크 (K열까지 필요 -> index 10이 존재해야 함)
    if df.shape[1] <= COL_K_ORDERNO:
        raise ValueError(f"엑셀 열 개수가 부족합니다. 현재 {df.shape[1]}열인데 K열(11번째 열)이 필요합니다.")

    # 필요한 컬럼만 뽑아서 정리
    a = _clean_series(df.iloc[:, COL_A_NAME])
    c = _clean_series(df.iloc[:, COL_C_PLATFORM])
    d = _clean_series(df.iloc[:, COL_D_PRODUCT])
    k = _clean_series(df.iloc[:, COL_K_ORDERNO])

    work = pd.DataFrame({
        "이름(A)": a,
        "플랫폼명(C)": c,
        "제품코드(D)": d,
        "주문코드(K)": k,
    })

    # 유효행: 이름/제품코드/주문코드가 있어야 함
    valid = (work["이름(A)"] != "") & (work["제품코드(D)"] != "") & (work["주문코드(K)"] != "")
    work = work[valid].copy()

    if work.empty:
        return work.iloc[0:0]

    # (이름, 제품코드) 그룹에서 주문코드가 여러 개인지 확인
    k_nunique = work.groupby(["이름(A)", "제품코드(D)"])["주문코드(K)"].transform("nunique")

    bad = work[k_nunique >= 2].copy()
    bad.sort_values(["이름(A)", "제품코드(D)", "주문코드(K)"], inplace=True)

    # 화면에 띄우라는 3개 컬럼만 반환 (A 이름, C 플랫폼명, K 주문코드)
    return bad[["이름(A)", "플랫폼명(C)", "주문코드(K)"]].copy()


class ForbiddenDialog(QDialog):
    def __init__(self, bad_df: pd.DataFrame, parent=None):
        super().__init__(parent)
        self.setWindowTitle("수령인 검수")
        self.resize(900, 520)

        layout = QVBoxLayout(self)

        title = QLabel("수령인과 제품코드를 확인해주세요")
        title.setStyleSheet("font-size: 18px; font-weight: 800;")
        layout.addWidget(title)

        sub = QLabel("동일 수령인인데 주문코드가 다른 건이 있습니다. 확인해주세요.(합포장 금지)")
        sub.setStyleSheet("color:#555;")
        layout.addWidget(sub)

        table = QTableWidget(self)
        table.setRowCount(len(bad_df))
        table.setColumnCount(len(bad_df.columns))
        table.setHorizontalHeaderLabels(bad_df.columns.tolist())

        for r in range(len(bad_df)):
            for c, col in enumerate(bad_df.columns):
                val = "" if pd.isna(bad_df.iat[r, c]) else str(bad_df.iat[r, c])
                item = QTableWidgetItem(val)
                item.setFlags(item.flags() ^ Qt.ItemIsEditable)
                table.setItem(r, c, item)

        table.setSortingEnabled(True)
        table.resizeColumnsToContents()
        layout.addWidget(table)

        btn = QPushButton("확인")
        btn.clicked.connect(self.accept)
        layout.addWidget(btn)


def main():
    # ✅ 여기만 네 파일로 바꾸면 됨 (샘플 파일)
    excel_path = str(Path.home() / "Downloads" / "shoppling_order_20260227153941.xlsx")

    # ✅ 노트북/재실행 대응: QApplication 중복 생성 방지
    app = QApplication.instance() or QApplication(sys.argv)

    bad_df = check_merge_forbidden(excel_path)

    if bad_df.empty:
        QMessageBox.information(None, "수령인 검수", "검수결과 이상 없음")
        return

    dlg = ForbiddenDialog(bad_df)
    dlg.exec()


if __name__ == "__main__":
    main()